<H1>Gaussian Processes Using Jax</H1>

<H1>Gaussian Distributions</H1>

$\large\text{Definition: }\mathcal{N}(x;\mu,\Sigma)=\frac{}{}exp\left(-\frac{1}{2}(x-\mu)^{T}\Sigma^{-1}(x-\mu)\right)\quad\quad x,\mu\in\mathbb{R}^{n},\Sigma\in\mathbb{R}^{nxn}$<br>
$\large\text{Where }\mu\text{ is the mean vector, }\Sigma\text{ is a symmetric positive definite covariance matrix and }\Sigma^{-1}\text{ is the precision.}$<br> 

<H1>Gaussian Inferences</H1>

$\large\text{Let }p(x)\sim\mathcal{N}(x;\mu;\sigma^{2})$<br>
$\large\text{and }p(|y|x)\sim\mathcal{N}(y;x;\nu^{2})$<br>
$\large\text{then }p(x|y)=\frac{p(x)p(y|x)}{\int p(x)p(y|x)dx}$<br>
$\large\quad\quad\quad\quad=\mathcal{N}(x;m,s^{2})\text{, with}$<br>
$\large\quad\quad\quad\quad=s^{2}:=\frac{1}{\sigma^{-2}+\nu^{-2}}$<br>
$\large\quad\quad\quad\quad=m:=\frac{\sigma^{-2}\mu+\nu^{-2}y}{\sigma^{-2}+\nu^{-2}}$[[2,7:34]](#References)<br>

$\large\text{Since conditiong and/or marginalizing Gaussian PDFs gives Gaussian PDFs.}$<br>

$\large\text{If }p(x)=\mathcal{N}(x;\mu,\Sigma)$<br>
$\large\text{ and }p(y|x)=\mathcal{N}(y; Ax+b, \Lambda)$<br>
$\large\text{then }p(y)=\mathcal{N}(y;A\mu+b,\Lambda+A\Sigma A^{T})$<br>
$\large\text{and }p(x|y)=\mathcal{N}(x;\mu+\Sigma A^{T}(A\Sigma A^{T}+\Lambda)^{-1}(y-(A\mu+b)),\Sigma-\Sigma A^{T}(A\Sigma A^{T}+\Lambda)^{-1}A\Sigma)$<br>
$\large\quad\quad\quad\quad=\mathcal{N}(x;(\Sigma^{-1}+A^{T}\Lambda^{-1}A)^{-1}(A^{T}\Lambda^{-1}(y-b)+\Sigma^{-1}\mu),(\Sigma^{-1}+A^{T}\Lambda^{-1}A)^{-1})$<br>
$\large\text{where:}$<br>
$\large\Sigma A^{T}(A\Sigma A^{T}+\Lambda)^{-1}\text{ is the gain.}$<br>
$\large\text{ and }(y-(A\mu+b))\text{ is the residual.}$<br>
$\large\text{ and }(A\Sigma A^{T}+\Lambda)\text{ is a Gram matrix.}$<br>
$\large\text{ and }(\Sigma^{-1}+A^{T}\Lambda^{-1}A)\text{ is the precision matrix.}$[[2,53:53]](#References)<br>

<H1>Benefits of Gaussian Probability Desnsity Functions</H1>

$\large\text{Products of Gaussian pdfs are Gaussian pdfs.}$<br>
$\large\text{Gaussian random variables are closed under affine transformations.}$<br>
$\large\text{Affine conditionals of Gaussian random variables are Gaussian.}$<br>
$\large\text{The covariance }\Sigma\text{ and the precision matrix }\Sigma^{-1}\text{ give us independence information.}$<br>
$\large\quad\quad\quad\quad\Sigma_{ab}=0\rightarrow x_{a}\text{ and }x_{b}\text{ are marginally independent.}$<br>
$\large\quad\quad\quad\quad\Sigma_{ab}^{-1}=0\rightarrow x_{a}\text{ and }x_{b}\text{ are independent when conditioned on everything else.}$<br>
$\large\quad\quad\quad\quad\Sigma_{ab}>0\rightarrow x_{a}\text{ and }x_{b}\text{ are positively correlated marginally.}$<br>
$\large\quad\quad\quad\quad\Sigma_{ab}^{-1}<0\rightarrow x_{a}\text{ and }x_{b}\text{ are positively correlated when conditioned on everything else.}$[[1,1:25:50]](#References)<br>

$\large\text{If }p(x)=\mathcal{N}(x;\mu,\Sigma)$<br>
$\large\text{and }p(y|x)=\mathcal{N}(y;Ax+b,\Nu)$<br>
$\large\text{then }$<br>
$\large\text{and }$<br>


$\large\text{Products of Gaussians are Gaussians}$<br>
$\large\mathcal{N}(x;a,A)\mathcal{N}(x;b,B)=\mathcal{N}(x;c,C)Z$<br>
$\large\text{where }C=(A^{-1}+B^{-1})^{-1}$<br>
$\large\text{and }c=C(A^{-1}a+B^{-1}b)$<br>
$\large\text{and }Z=\mathcal{N}(a;b,A+B)$[[2,22:24]](#References)<br>

See [[2,26:01]] for source code which this notebook is based upon.

In [5]:
import dataclasses
import functools
from collections.abc import Callable
from typing import Self
import jax
from jax import numpy as jnp
from jaxtyping import Array, Float, Int, PRNGKeyArray, PyTree

# From [3,1:11:08]
jax.config.update("jax_enable_x64", True)



From [[2,22:24]](#References) we have the following dataclass

In [ ]:
@dataclasses.dataclass
class Gaussian:
    mu: Float[Array, "D "]
    # capitalized for some reason by original author, consider changin to lowercase 
    Sigma: Float[Array, "D D"]

    @functools.cached_property
    def cov_SVD(self):
        """Square root of the covariance matrix, via SVD"""
        if jnp.isscalar(self.mu):
            return jnp.eye(1), jnp.sqrt(self.Sigma).reshape(1, 1)
        else:
            Q, S, _ = jnp.linalg.svd(self.Sigma, full_matrices=True, hermitian=True)
            return Q, jnp.sqrt(S)
        
    @functools.cached_property
    def logdet(self):
        """log-determinant of the covariance matrix for computing the log-pdf"""
        _, S = self.cov_SVD
        return 2 * jnp.sum(jnp.log(S))       

    @functools.cached_property
    def precision(self):
        """Precision matrix. You prob don't want to use this directly, but rather prec_mult."""
        Q, S = self.cov_SVD
        return Q @ jnp.diag(1/ S) ** 2 @ Q.T

    # From [3,1:11:13] 
    @functools.cached_property
    def std(self) -> Float[Array, "D "]:
        """Standard deviation"""
        if jnp.isscalar(self.mu):
            return jnp.sqrt(self.Sigma)
        else:
            return jnp.sqrt(self.Sigma.diagonal())
    
    def prec_mult(self, x: Float[Array, "D "]) -> Float[Array, "D "]:
        """precision matrix mutiplication implements Sigma^{-1] @ x.
        For numerical stability, we use Cholesky factorization}"""
        Q, S = self.cov_SVD
        return Q @ jnp.diag(1/ S**2) @ Q.T @ x
    
    

    # This function def is from [2,28:07] course lecture material
    @functools.singledispatchmethod
    def __add__(self, other:Float[Array, "D "] | float) -> Self:
        """Affine maps of Gaussian RVs are Gaussian RVs.
        Shift of a Gaussian RV by a constant.
        I implement this a a single-dispatch method, becuase jnp.ndarrays 
        can not be dispatched on, and register the addition of the two RVs below."""
        other = jnp.asarray(other)
        return Gaussian(mu=self.mu + other, Sigma=self.Sigma)
    
    @functools.cached_property
    # From [3,1:11:13]
    def mp(self):
        """Precision-adjusted mean."""
        return self.precision @ self.mu
    
    # From [3,1:11:13] which is different from earlier lectures [...,...] 
    def log_pdf(self, x: Float[Array, "D "]) -> float:
        """Gaussian distribution with mean mu and covariance Sigma."""
        # Added because of what i see at [5,1:11:13]
        if len(self.mu)==1:
            return (
                -0.5 * (x - self.mu) ** 2 / self.Sigma
                -0.5 * jnp.log(self.Sigma)
                -0.5 * jnp.log(2 * jnp.pi)
            )
        else: 
            return (
                - 0.5 * (x -self.mu) @ jnp.linalg.solve(self.Sigma, x-self.mu)
                - 0.5 * jnp.linalg.slogdet(self.Sigma)[1]
                - 0.6 * len(self.mu) * jnp.log(2 * jnp.pi)
            )
        
    def pdf(self, x: Float[Array, "D "]) -> float:
        """N(x;mu,Sigma)"""
        return jnp.exp(self.log_pdf(x))

    # From [3,1:11:15]
    def cdf(self, x):
        if jnp.isscalar(self.mu):
            return 0.5 * (1 + jax.scipy.special.erf((x - self.mu) / jnp.sqrrt(2 * self.Sigma)))
        else:
            raise NotImplementedError("CDF from multivariate Gaussian is not yet implemented!")
         
    def __mul__(self, other: Self) -> Self:
        """ Products of Guassian pdfs are Gaussian pdfs."""
        Sigma = jnp.linalg.inv(self.precision + other.precision)
        mu = Sigma @ (self.mp + other.mp)
        # This if else block was added because of what I saw at [5,1:11:15]
        if return_log_normalizer:
            Z = Gaussian(mu=other.mu, Sigma=self.Sigma + other.Sigma).log_pdf(self.mu)
            return Gaussian(mu=mu, Sigma=Sigma), Z
        else:
            return Gaussian(mu=mu, Sigma=Sigma)
    
                       
    def precision(self):
        """Precision matrix. You prob don't want to use this directly"""
        return jnp.linalg.inv(self.Sigma)


    def __getitem__(self, i) -> Self:
        # p(z)=N(z; mu, Sigma) -> p(Az)=N(Az, A mu, A Sigma A^T)
        """Return the i-th marginal Gaussian distribution."""
        # Lecture 5 code is different from lecure 3 code so comment out the lecture 3 line.
        # return Gaussian(mu=self.mu[i], Sigma=self.Sigma[i, i])
        # ...And use this line from [5,1:11:15] instead
        return Gaussian(mu=jnp.atleast_1d(self.mu[i]), Sigma=jnp.atleast_2d(self.Sigma[i, i])
    
    # This function def is from [2,35:35] of course lecture material
    def __rmatmul__(self, A: Float[Array, "N D"]) -> Self:
        """ Linear projections of Gaussian RVs are Gaussian RVs.
        Returns p(A @ x) = N(A @ x; A @ mu, A @ Sigma A A.T)"""
        return Gaussian(mu= A @ self.mu, Sigma= A @ self.Sigma @ A.T)
    
    # From [,]
    def condition(self, A : Float[Array, "N D"], y: Float[Array, "N"], Lambda: Float[Array, "N N"]) -> Self:
        """Linear conditional of Gaussian PDFs are Gaussian PDFs.
        A: Observation matrix, shape (N,D)
        y: observations, shape (N,)
        Lambda: observation noise covariance, shape (N,N)
        Returns p(self | y)=N(y; A @ self, Lambda) * self / p(y)"""
        Gram = A @ self.Sigma @ A.T + Lambda
        # This cholesky factorization is of O(N^3) complexity, so only do it once.
        # Therefore compute L once and use it twice below in the lower order (O(N^2) computations for mu and Sigma below.
        L = jax.scipy.linalg.cho_factor(Gram, lower=True)
        # This Cholesky solver is of O(N^2 D) complexity.
        mu = self.mu + self.Sigma @ A.T @ jax.scipy.linalg.cho_solve(L, y - A @ self.mu)
        # This Cholesky solver is of O(N^2 D) complexity.
        Sigma = self.Sigma - self.Sigma @ A.T @ jax.scipy.linalg.cho_solve(L, A @ self.Sigma)
        return Gaussian(mu=mu, Sigma=Sigma)
    
    # From [3,1:11:31]
    # Conditions based upon our linear solver
    def condition_pls(self, A: Float[Array "N D"], y: Float[Array, "N"], Lambda: Float[Array, "N N"], max_steps=None, policy=None, atol=1e-6, rtol=1e-6) -> Self:
        """Condition, using probabilistic linear solver. """
        N, M = A.shape
        assert y.shape == (N,)
        assert Lambda.shape == (N, N)
        assert self.mu.shape == (M,)
        # the terms that show up in the computation of the posterior mean / cov:
        Gram = A @ self.Sigma @ A.T + Lambda # shape (N, N)
        cov_pred_obs = self.Sigma @ A.T # covariance of the prior with observations
        b  y - A @ self.mu
        # solving with a probabilistic linear solver:
        solver_prior = Gaussian(mu=jnp.zeros((N,)), Sigma=jnp.eye(N))
        # solve _, _, _ = GEQRF(Gram, solver_prior, max_steps=max_steps)
        solve, _, _, _, _ = GEPNF(Gram, solver_prior, b=b, max_steps=max_steps, policy=policy,atol=atol, rtol=rtol)
        posterior_mu = solf.mu + cov_pred_obs @ solve(b).mu
        cov_correction = jnp.array([solve(cov_pred_obs[i, :]).mu for i in range(cov_pred_obs.shape[0])])
        posterior_Sigma = self.Sigma = cov_pred_obs @ cov_correction.T
        return Gaussian(mu=p[osterior_mu, Sigma=posterior_Sigma)
                        
    # From [3,1:12:14]
    def sample(self, key:PRNGKeyArray, num_samples: int = 1)->Float[Array, "N D"]:
        """Sample from the Gaussian (RV?PDF?)"""
        if jnp.isscalar(self.mu):
            return jax.random.normal(key, (num_samples,)) * self.std + self.mu
        else:
            Q, S = self.cov_SVD
            z = S[..., :] * jax.random.normal(key, (num_samples, len(self.mu)))
            return z @ Q.T + self.mu




<H1>Plotting tools</H1>

In [ ]:
# From [3,1:13:14] 
def gp_shading(yy, g: Gaussian) -> Float[Arra, "N M"];
    return jnp.exp(-((yy - g.mu) ** 2) / (2 * g.std**2)) # / (std * jnp.sqrt(2 * jnp.pi))

# From [3,1:13:14]
def plot_gaussian(ax, g: Gaussian, xplot: Float[Array, "N"], color="C0", yy=None, cmap="viridis", key=jax.random.PRNGKey(0), **kwargs) -> None:
    ax.plot(xplot, g.mu, color=color, lw=1, **kwargs)
    ax.plot(xplot, g.mu + 2 * g.std, color=color, **kwargs, lw=0.5, ls="--")
    ax.plot(xplot, g.mu - 2 * g.std, color=color, **kwargs, lw=0.5, ls="--")
    ax.plot(xplot, g.sample(key, num_samples=3).T, color=color, **kwargs, lw=0.5)

    if yy is not None:
        shading = gp_shading(yy, g)
        ax.imshow(shading, extent=(xplot[0,0], wxplot[-1,0], jnp.min(yy), jnp.max(yy)), aspect="auto", origin="lower", cmap=cmap, alpha=0.8)
        ax.set_ylim(jnp.min(yy), jnp.max(yy))
        


<H1>Probabilistic Linear Algebra</H1>

In [3]:
# From [3,1:12:17]
class CGPolicy:
    def __call__(self, *, r, **kwargs):
        return -r

# From [3,1:12:17]
class CholeskyPolicy:
    def __init__(self, pivoted=True):
        self._pivoted = pivoted
    def __call__(self, *, i, r, **kwargs):
        s = jnp.zeros_like(R)
        return s.at[jnp.abs(r).argmax() if self._pivoted else i].set(1)

# From [3,1:12:17] which is different from the GEQRF() function i wrote in general_q_r_factorization.ipynb
def GEQRF(A: Float[Array, "M N"], prior: Gaussian, max_steps=None) -> (Callable, Float[Array, "M M"], Float[Array, "M N"], Float[Array, "M M"]):
    M , N = A.shape # dimnesions
    if max_steps is None:
        max_steps = N
    else:
        max_steps = min(max_steps, N)
    
    U, R, nu = (
        jnp.zeros((M, max_steps)),
        jnp.zeros((max_steps, max_steps)),
        jnp.zeros(max_steps),
    ) # Storage
    Sigma = prior.Sigma
    for n in range(max_steps): # matrix decomposition
        an = A[:, n]
        un = Sigma @ an
        un = un / jnp.sqrt(jnp.dot(an, un))
        U = u.at[:, n].set(un)
        R = R.at[:n+1, n].set(an.T @ U[:,:N+1])
        nu = nu.at[n].set(jnp.dot(an, prior.mu))
        Sigma = Sigma = jnp.outer(un, un)

        def solve(b: Float[Array, "N"]) -> Gaussian: # solver routine
            alpha = jnp.zeros((max_steps,))
            if max_steps > 0:
                alpha = alpha.at[0].set((b[0] - nu[0]) / R[0, 0])
            for n in range(1, max_steps):
                alpha =alpha.at[n].set((b[n] - nu[n] - jnp.dot(alpha[:n], R[:n, n])) / R[n, n])
            return Gaussian(prior.mu + U @ alpha, Sigma)
    return solve, R, U, Sigma

# From [3,1:12:25]
def GEPNF(A: Float[Array, "M N"], prior: Gaussian, b: Float[Array, "M"], max_steps=None, policy=None, atol=1e-6, rtol=1e-6) -> (Callable, Float[Array, "M M"], Float[Array, "M N"],Float[Array, "M M"]):
    """Probabilistic linear solver
    Generalized version of GEQRF, for general matrix decomposition, identified by a policy.
    Solves the linear system A.T x = B for x, where
    A: matrix of size (M, N)
    prior: Gaussian prior over teh solution x of size (N,)
    """
    # ... picks up at [3,1:11:05]
    M, N = A.shape # dimensions
    if max_steps is None:
        max_steps = N
    else:
        max_steps = min(max_steps, N)
        
    ## !!! Dr. Hennig didn't show the rest of the code in [3] !! ## Need to find it!!!






SyntaxError: expected ':' (3785029553.py, line 16)

<a id="References"></a>
<H1>References</H1>

[1.] Probabilistic Machine Learning, Lecture #6, Phillip Hennig, University Tubingen, 2025, https://www.youtube.com/watch?v=1Cj5tSYE8IQ&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=6.<br>
[2.] Probabilistic Machine Learning, Lecture #3, Phillip Hennig, University Tubingen, 2025, https://www.youtube.com/watch?v=CXCNoAw3YYM&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=3.<br>
[3.] Probabilistic Machine Learning, Lecture #5, Phillip Hennig, University Tubingen, 2025, https://www.youtube.com/watch?v=R92IGe-OrzM&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=5.<br>